### Text Summarization & Entity Recognition using Transformers and spaCy

In [17]:
!pip install transformers datasets spacy torch torchvision torchaudio pandas matplotlib sentencepiece rouge-score
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [18]:
import pandas as pd
import numpy as np
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import spacy
import matplotlib.pyplot as plt
from datasets import load_dataset
from rouge_score import rouge_scorer

In [19]:
# -- Load dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")
train_df = pd.DataFrame(dataset['train'].select(range(2000))) # take only 2000 rows are enough to demonstrate model fine-tuning


In [20]:
train_df.head()


,article,highlights,id
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,42c027e4ff9730fbb3de84c1af0d2c506e41c3e4
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...,ee8871b15c50d0db17b0179a6d2beab35065f1e9
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa...",06352019a19ae31e527f37f7571c6dd7f0c5da37
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non...",24521a2abb2e1f5e34e6824e0f9e56904a2b0e88
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical...",7fe70cc8b12fab2d0a258fababf7d9c6b5e1262a


In [21]:
train_df = train_df[['article', 'highlights']] # drop id field
train_df.head()

,article,highlights
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa..."
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non..."
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical..."


In [22]:
train_df.shape

(2000, 2)

In [58]:
# -- Load the Pretrained Summarization Model # facebook/bart-large-cn took very long
# "facebook/bart-base"  ->  half the size of large version
# "google/pegasus-xsum"  ->  short summaries, faster
# "t5-small"  ->  compact and versatile

model_name = "google/pegasus-xsum"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

summarizer = pipeline("summarization", model=model, tokenizer=tokenizer)

Some weights of PegasusForConditionalGeneration were not initialized from the model checkpoint at google/pegasus-xsum and are newly initialized: ['model.decoder.embed_positions.weight', 'model.encoder.embed_positions.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cpu


In [59]:
# test summarization
sample_text = train_df.iloc[0,0]
print("Original Text:\n", sample_text[:800])

summary = summarizer(sample_text, max_length=100, min_length=30, truncation=True, do_sample=False)
print("\nGenerated Summary:\n", summary[0]['summary_text'])

Original Text:
 LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18,

Generated Summary:
 "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar."


In [60]:
# -- training data: highlights[0]
print(train_df.iloc[0,1])

Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


In [61]:
# -- Evaluate Summarization Quality (ROUGE / BLEU)
# ROUGE is a set of metrics and a software package used for evaluating automatic summarization and machine translation software in natural language processing.
# rougeL - It takes the longest common subsequences (LCS)
from tqdm import tqdm

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
scores = []
summaries = []

for i in tqdm(range(10)): # summarize for 10 samples
    ref = train_df['highlights'][i]
    pred = summarizer(train_df['article'][i], max_length=100, min_length=30, truncation=True, do_sample=False)[0]['summary_text']
    score = scorer.score(ref, pred)
    scores.append(score['rougeL'].fmeasure)
    summaries.append(pred)

print(f"Average ROUGE-L: {np.mean(scores):.2f}")

100%|██████████| 10/10 [04:22<00:00, 26.29s/it]

Average ROUGE-L: 0.17


**Results Summary**
- Average ROUGE-L Score: 0.17


In [62]:
# -- Visualize Entity Distribution
# -- Named Entity Recognition (NER) with spaCy
# "en_core_web_sm" -> small, fast, less accurate
# "en_core_web_md"  -> medium, more accurate, larger
# "en_core_web_lg"  -> large, most accurate, largest
nlp = spacy.load("en_core_web_sm")
def extract_entities(text):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    for ent in doc.ents:
      print(f"{ent.text:<25} | {ent.label_}")
    return entities

entities = extract_entities(summaries[3])
print("\nSummary:\n", summary)
print("\nEntities:\n", entities)

George W. Bush            | PERSON
Camp David                | FAC
Maryland                  | GPE
the White House           | ORG
the National Naval Medical Center | ORG
Bethesda                  | GPE
Maryland                  | GPE
Scott Stanzel             | PERSON

Summary:
 [{'summary_text': '"I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar."'}]

Entities:
 [('George W. Bush', 'PERSON'), ('Camp David', 'FAC'), ('Maryland', 'GPE'), ('the White House', 'ORG'), ('the National Naval Medical Center', 'ORG'), ('Bethesda', 'GPE'), ('Maryland', 'GPE'), ('Scott Stanzel', 'PERSON')]


In [63]:
with open("./output.txt", "w") as f:
  for i in tqdm(range(10)): # entity extract for 10 samples
    entities = extract_entities(summaries[i])
    f.write(f"{summaries[i]};{entities}\n")

100%|██████████| 10/10 [00:00<00:00, 60.68it/s]

18                        | DATE
Miami                     | GPE
Miami                     | GPE
the Mississippi River     | LOC
Wednesday                 | DATE
afternoon                 | TIME
at least eight            | CARDINAL
more than 100             | CARDINAL
Mississippi               | LOC
Gary Babineau             | PERSON
CNN                       | ORG
CNN                       | ORG
George W. Bush            | PERSON
Camp David                | FAC
Maryland                  | GPE
the White House           | ORG
the National Naval Medical Center | ORG
Bethesda                  | GPE
Maryland                  | GPE
Scott Stanzel             | PERSON
Atlanta Falcons           | ORG
Michael Vick              | PERSON
up to five years          | DATE
Youssif al-Hakim's        | PERSON
the United States         | GPE
Iraq                      | GPE
Iraqi                     | NORP
Baghdad                   | GPE
Colombia                  | GPE
the Revolutionary Armed Forces of 